# BE-embedding notebook

This notebook provides a modular interface for using the **BE-embedding**.

The running example is directed and weighted, so it shows all the possible settings of our approach

## Input format

The input network must be an edgelist file.

Unweighted network:

```text
source target
```

Weighted network:

```text
source target weight
```

Node ids must start from `0` and must be consecutive up to `N-1`.

## Setup

In [1]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd

ROOT = Path.cwd()
JAR_PATH = ROOT / "epsBE.jar"

print("Repository root:", ROOT)
print("epsBE.jar found:", JAR_PATH.exists())

Repository root: /home/squillace/Giuseppe/EmbNetworks
epsBE.jar found: True


## Network parameters

The default example uses the small test network in `Test/test.edgelist`. Change these values to use another network.

In [2]:
net_path = "Test/test.edgelist"
net_t_path = "Test/testT.edgelist"
n_nodes = 3

# The example is directed, so both A and A^T are used.
directed = True
weighted = True

partition_path = "embed/example_BE"
partition_t_path = "embed/example_TBE"
embedding_path = "embed/example_EMB.csv"

## eps-BE parameters

noise is just to handle the precision error, is not mandatory

In [3]:
noise = 0.000001
eps_0 = 0.1 + noise
delta_max = 0.1 + noise
delta_step = 1

## Utility functions

In [4]:
def path_from_root(path):
    return ROOT / path if isinstance(path, str) else Path(path)


def bool_for_jar(value):
    return "true" if value else "false"


def read_edgelist_matrix(net_path, n_nodes, directed, weighted):
    net_path = path_from_root(net_path)
    matrix = np.zeros((n_nodes, n_nodes), dtype=float)

    with net_path.open("r") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("%"):
                continue
            parts = line.split()
            source = int(parts[0])
            target = int(parts[1])
            value = float(parts[2]) if weighted else 1.0
            matrix[source, target] = value
            if not directed:
                matrix[target, source] = value

    return matrix


def run_eps_be(net_path, n_nodes, eps_0, delta_max, delta_step, partition_path, directed, weighted):
    net_path = path_from_root(net_path)
    partition_path = path_from_root(partition_path)
    partition_path.parent.mkdir(parents=True, exist_ok=True)

    if not JAR_PATH.exists():
        raise FileNotFoundError("epsBE.jar was not found in the repository root")
    if not net_path.exists():
        raise FileNotFoundError("Network file not found: " + str(net_path))

    command = [
        "java", "-jar", str(JAR_PATH), str(net_path), str(n_nodes),
        str(eps_0), str(delta_max), str(delta_step), str(partition_path),
        bool_for_jar(directed), bool_for_jar(weighted),
    ]

    result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr or result.stdout)

    return partition_path


def read_partition(partition_path):
    partition_path = path_from_root(partition_path)
    lines = partition_path.read_text().splitlines()
    if len(lines) < 2:
        raise ValueError("Invalid partition file: " + str(partition_path))

    partition = []
    for block_text in lines[1].split(","):
        block = []
        for token in block_text.split():
            if token.startswith("x"):
                block.append(int(token.replace("x", "")))
        if block:
            partition.append(block)

    return partition


def reduce_by_partition(matrix, partition):
    embedding = np.zeros((matrix.shape[0], len(partition)), dtype=float)
    for column, block in enumerate(partition):
        indices = [node_id - 1 for node_id in block]
        embedding[:, column] = matrix[:, indices].sum(axis=1)
    return embedding


def compute_embedding_block(matrix, partition_path):
    partition = read_partition(partition_path)
    return reduce_by_partition(matrix, partition)


def save_embedding(embedding, embedding_path):
    embedding_path = path_from_root(embedding_path)
    embedding_path.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(embedding_path, embedding, delimiter=" ", fmt="%.6f")
    return embedding_path

## Inspect the test adjacency matrices

In [5]:
A = read_edgelist_matrix(net_path, n_nodes, directed, weighted)
AT = read_edgelist_matrix(net_t_path, n_nodes, directed, weighted)

A_df = pd.DataFrame(A, index=range(n_nodes), columns=range(n_nodes))
AT_df = pd.DataFrame(AT, index=range(n_nodes), columns=range(n_nodes))

print("Adjacency matrix A:")
display(A_df)

print("Adjacency matrix A^T, loaded from its edgelist file:")
display(AT_df)

Adjacency matrix A:


,0,1,2
0,0.0,0.3,0.4
1,0.0,0.0,0.0
2,0.0,0.0,0.0


Adjacency matrix A^T, loaded from its edgelist file:


,0,1,2
0,0.0,0.0,0.0
1,0.3,0.0,0.0
2,0.4,0.0,0.0


## Step 1: compute the $\varepsilon$-BE partition of A

This cell runs the eps-BE algorithm on the edgelist representation of the original adjacency matrix `A`.

Recall that the implementation processes outgoing edges while BE consider the incoming edges. This distinction is irrelevant for undirected networks, where incoming and outgoing neighborhoods coincide. 

For directed networks, instead, BE on `A` is obtained by running the algorithm on the transpose matrix `A^T` while BE on `A^T` is obtained by running the algorithm on the transpose matrix `A`.

In [12]:
run_eps_be(
    net_path=net_t_path,
    n_nodes=n_nodes,
    eps_0=eps_0,
    delta_max=delta_max,
    delta_step=delta_step,
    partition_path=partition_t_path,
    directed=directed,
    weighted=weighted,
)

partition = read_partition(partition_path)

print("Partition 0.1-BE of A saved to:", path_from_root(partition_path))
print("Partition 0.1-BE of A:")
partition

Partition 0.1-BE of A saved to: /home/squillace/Giuseppe/EmbNetworks/embed/example_BE
Partition 0.1-BE of A:


[[2, 3], [1]]

## Step 1b: compute the BE partition of A^T

This cell runs the same eps-BE algorithm on the edgelist representation of `A^T`. The file `net_t_path` must already exist; the notebook does not compute or write the transpose edgelist.

For directed networks, this partition is needed to build the second embedding block before concatenation.


In [13]:
run_eps_be(
    net_path=net_path,
    n_nodes=n_nodes,
    eps_0=eps_0,
    delta_max=delta_max,
    delta_step=delta_step,
    partition_path=partition_path,
    directed=directed,
    weighted=weighted,
)

partition_t = read_partition(partition_t_path)

print("Partition 0.1-BE of A^T saved to:", path_from_root(partition_t_path))
print("Partition 0.1-BE of A^T:")
partition_t

Partition 0.1-BE of A^T saved to: /home/squillace/Giuseppe/EmbNetworks/embed/example_TBE
Partition 0.1-BE of A^T:


[[1], [2, 3]]

## Step 2: compute and concatenate the embedding blocks

The embedding block from `A` is obtained by reducing `A` with the partition computed from `A`. The embedding block from `A^T` is obtained by reducing `A^T` with the partition computed from `A^T`. The final directed embedding is the column-wise concatenation of the two blocks.


In [15]:
embedding_A = compute_embedding_block(A, partition_path)
embedding_AT = compute_embedding_block(AT, partition_t_path)
embedding = np.concatenate([embedding_AT, embedding_A], axis=1)
saved_path = save_embedding(embedding, embedding_path)

embedding_A_df = pd.DataFrame(embedding_A, index=pd.RangeIndex(n_nodes, name="node"))
embedding_AT_df = pd.DataFrame(embedding_AT, index=pd.RangeIndex(n_nodes, name="node"))
embedding_df = pd.DataFrame(embedding, index=pd.RangeIndex(n_nodes, name="node"))

print("Embedding of A:")
display(embedding_A_df)

print("Embedding of A^T:")
display(embedding_AT_df)

print("Final embedding matrix:")
display(embedding_df)

Embedding of A:


,0,1
node,,
0,0.7,0.0
1,0.0,0.0
2,0.0,0.0


Embedding of A^T:


,0,1
node,,
0,0.0,0.0
1,0.3,0.0
2,0.4,0.0


Final embedding matrix:


,0,1,2,3
node,,,,
0,0.0,0.0,0.7,0.0
1,0.3,0.0,0.0,0.0
2,0.4,0.0,0.0,0.0
